# Exercise 1 – Task: people and text in three new images

**Goal:** for the images `parking.jpg`, `sheep.jpg` and `street.jpg`, build a table with the fields `scene`, `people_count` and `text_in_image`, and score it against the hand labels.

Build it yourself in three steps. Every step can be done by **copying code from `image_extraction_example.ipynb` and adapting it**.

Each step: write your code in the empty cell → run the **check** (✓ go on, ✗ shows what is missing; a `NameError` means the variable does not exist yet). Stuck or out of time? Open the **hint**, then the **solution**.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret. This notebook uses the **OpenAI API** (model `gpt-6-luna`); other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code.

In [ ]:
# Python libraries.
import sys
from typing import Literal

import jiwer
import pandas as pd
from pydantic import BaseModel
from sklearn.metrics import accuracy_score

# helper.py and checks.py are in the main folder, one level up.
sys.path.append("..")
from helper import DATA, ask, show_images
from checks import check_extracted, check_record, check_scores

files = ["parking.jpg", "sheep.jpg", "street.jpg"]
show_images([DATA / "images" / name for name in files])

## Step 1: answer format and prompt

Write a class `ImageRecord` and a `PROMPT` for these fields, then test them on `street.jpg` and store the result in `answer`.

| Field | Type | Instruction for the model |
| --- | --- | --- |
| `scene` | one of the four scenes | Pick the scene from the list. |
| `people_count` | whole number (`int`) | Count the visible people. |
| `text_in_image` | text, or `None` (`str \| None`) | Copy the most prominent text. Use null if there is no text. Use [unreadable] if the text cannot be read. |

In [ ]:
# Your code here.


In [ ]:
check_record(answer)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 2: copy `LakeRecord` and its `PROMPT`. Replace the field `objects` and its instruction line with the two new fields.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
class ImageRecord(BaseModel):
    scene: Literal["lake", "parking lot", "alpine pasture", "city street"]
    people_count: int
    text_in_image: str | None


PROMPT = """
Pick the scene from the list.
Count the visible people.
Copy the most prominent text. Use null if there is no text.
Use [unreadable] if the text cannot be read.
"""

answer = ask(PROMPT, ImageRecord, [DATA / "images" / "street.jpg"])
answer
```

</details>

## Step 2: the table

Ask about every image in `files` and build the DataFrame `extracted`: one row per image, with a column `file`.

In [ ]:
# Your code here.


In [ ]:
check_extracted(extracted, files)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 3: the same loop – only with your `ImageRecord`.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
rows = []
for name in files:
    row = ask(PROMPT, ImageRecord, [DATA / "images" / name])
    row["file"] = name
    rows.append(row)

extracted = pd.DataFrame(rows)
extracted
```

</details>

## Step 3: score it

Compare `extracted` with the hand labels:

- `people`: the accuracy of `people_count`, like the scene accuracy in the example.
- `cer`: the character error rate of `text_in_image` – how many characters must be changed, per character of the label (lower is better). Use only the images whose label has a text: `scores[scores["text_in_image_ref"].notna()]`. Then `jiwer.cer(label_texts, model_texts)` with two lists; a missing model text counts as `""` (`.fillna("")`).

In [ ]:
# Your code here.


In [ ]:
check_scores(people, cer)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 4: the same merge. Then `accuracy_score` for the people and `jiwer.cer` for the text.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
labels = pd.read_csv(DATA / "images" / "labels.csv")
scores = extracted.merge(labels, on="file", suffixes=("_pred", "_ref"))

people = accuracy_score(scores["people_count_ref"], scores["people_count_pred"])

with_text = scores[scores["text_in_image_ref"].notna()]
cer = jiwer.cer(list(with_text["text_in_image_ref"]),
                list(with_text["text_in_image_pred"].fillna("")))

print("People accuracy:", round(people, 2), "– CER:", round(cer, 2))
```

</details>

**Extra:** add more images from `data/images/` to `files` and run steps 2 and 3 again.

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. Why can a perfectly valid JSON answer still be wrong? Look at the object lists in the example and at your people counts.
2. Which field has the lower score in your run, people or text? What makes it hard for the model?
3. In the example, the label says `tree` and the model answers `pine tree`. Is the model really wrong – and how could you score such answers fairly?
4. Why do we add the filename in Python instead of asking the model for it?
5. You want to extract 10,000 images. What do you check before you start the run?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. The JSON format only checks the form: all fields are there and have the right type. It does not check whether the content is true. A wrong people count or an invented object is still valid JSON.

2. Use your measured scores. Counting people and reading small text are often weak. The scene is easier, because the model only picks one of four choices.

3. An exact comparison counts it as wrong, although the answer is reasonable. A clearer prompt, a fixed list of object names, or cleaning the names before scoring helps.

4. Python knows the filename exactly. The model could misspell it, and then the row no longer matches its label. Do not ask the model for something you already know.

5. Estimate the cost first: tokens per image × number of images × price. Test the prompt on a small labelled sample, score each field, and keep the log of all answers.

</details>

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')